In [1]:
# Multi-Dataset Model Training, Pipeline Refactoring, & Cost Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import datetime
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve, auc
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import shap

In [2]:
# Create directories for saving artifacts
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)

In [3]:
# 1. Load Data
df = pd.read_csv('../data/raw/breast_cancer_prediction.csv')
print("Initial Shape:", df.shape)
df.head()

Initial Shape: (10000, 23)


,Patient_ID,Age,Gender,BMI,Family_History,Smoking,Alcohol_Consumption,Physical_Activity,Hormone_Therapy,Menopause_Status,...,Mammogram_Result,Biopsy_Result,Cancer_Stage,Cancer,Blood_Pressure,Cholesterol,Diabetes,Exercise_Days_Per_Week,Breastfeeding_History,Annual_Income_USD
0,100001,71,Female,27.2,Yes,Yes,Yes,High,No,Post,...,Suspicious,Malignant,Stage II,1,141,167,No,7,No,101316
1,100002,34,Male,26.0,No,No,No,High,No,Post,...,Normal,Benign,No Cancer,0,104,229,Yes,7,Yes,30023
2,100003,80,Female,20.7,No,No,No,High,No,Pre,...,Suspicious,Benign,No Cancer,0,161,275,No,1,Yes,34819
3,100004,40,Female,26.8,Yes,Yes,No,Moderate,No,Post,...,Normal,Benign,No Cancer,0,150,180,No,7,Yes,68025
4,100005,43,Female,28.6,Yes,Yes,No,Moderate,No,Post,...,Normal,Benign,No Cancer,0,110,266,No,0,Yes,90919


In [4]:
# 2. Data Cleaning
# Remove duplicates
df.drop_duplicates(inplace=True)

# Drop Patient_ID and leaky features
cols_to_drop = ['Patient_ID', 'Biopsy_Result', 'Cancer_Stage', 'Mammogram_Result', 'Lymph_Node_Involvement', 'Tumor_Size_cm']
for col in cols_to_drop:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)
df.dropna(inplace=True)

In [5]:
# 3. Train-Test Split (BEFORE preprocessing to prevent data leakage)
X = df.drop('Cancer', axis=1)
y = df['Cancer']

# Ensure target is integer
y = y.astype(int)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train_raw.shape}, Testing set: {X_test_raw.shape}")

Training set: (7760, 16), Testing set: (1940, 16)


In [6]:
# 4. Pipeline Definition (Preprocessing + Resampling)
categorical_cols = X_train_raw.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Preprocessing for numerical data
numerical_transformer = StandardScaler()

# Preprocessing for categorical data
# Use OrdinalEncoder with handle_unknown='use_encoded_value' to prevent errors on unseen data
categorical_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Bundle preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [7]:
# 5. Model Initialization
# We only use Baseline, LogReg, RF, and GradientBoosting as per requirements
models = {
    'Baseline (Dummy)': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

datasets = {
    'Imbalanced': None,
    'SMOTE': SMOTE(random_state=42),
    'Undersampled': RandomUnderSampler(random_state=42)
}

In [8]:
# 6. Evaluation Loop with 5-Fold Cross Validation
results = []
trained_pipelines = {}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for ds_name, sampler in datasets.items():
    trained_pipelines[ds_name] = {}
    for model_name, classifier in models.items():
        # Build pipeline
        steps = [('preprocessor', preprocessor)]
        if sampler is not None:
            steps.append(('sampler', sampler))
        
        # Calibrate the classifier to get better probability estimates, except for Dummy
        if model_name != 'Baseline (Dummy)':
            calibrated_clf = CalibratedClassifierCV(classifier, cv=cv, method='sigmoid')
            steps.append(('classifier', calibrated_clf))
        else:
            steps.append(('classifier', classifier))
            
        pipeline = ImbPipeline(steps=steps)
        
        # Train
        pipeline.fit(X_train_raw, y_train)
        
        # Predict on Test Set
        y_pred = pipeline.predict(X_test_raw)
        y_prob = pipeline.predict_proba(X_test_raw)[:, 1] if hasattr(pipeline, 'predict_proba') else None
        
        # Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        if y_prob is not None:
            roc_auc = roc_auc_score(y_test, y_prob)
            precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
            pr_auc = auc(recalls, precisions)
        else:
            roc_auc = 'N/A'
            pr_auc = 'N/A'
            
        results.append({
            'Dataset': ds_name,
            'Model': model_name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1 Score': f1,
            'ROC-AUC': roc_auc,
            'PR-AUC': pr_auc
        })
        trained_pipelines[ds_name][model_name] = pipeline

results_df = pd.DataFrame(results)
display(results_df.sort_values(by=['Recall', 'PR-AUC'], ascending=[False, False]))

,Dataset,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
11,Undersampled,Gradient Boosting,0.771649,0.453782,0.859416,0.593951,0.884769,0.685659
10,Undersampled,Random Forest,0.770103,0.448737,0.801061,0.575238,0.879786,0.672662
9,Undersampled,Logistic Regression,0.792784,0.479741,0.785146,0.595573,0.877945,0.688402
5,SMOTE,Logistic Regression,0.792784,0.479407,0.771883,0.591463,0.876862,0.686473
7,SMOTE,Gradient Boosting,0.863918,0.673846,0.580902,0.623932,0.881779,0.693741
6,SMOTE,Random Forest,0.868041,0.693291,0.575597,0.628986,0.878194,0.684098
3,Imbalanced,Gradient Boosting,0.876804,0.739583,0.564987,0.640602,0.882637,0.691253
2,Imbalanced,Random Forest,0.870619,0.742308,0.511936,0.605965,0.880353,0.692542
1,Imbalanced,Logistic Regression,0.862887,0.736170,0.458886,0.565359,0.879741,0.689974
0,Imbalanced,Baseline (Dummy),0.805670,0.000000,0.000000,0.000000,0.500000,0.597165


In [9]:
# 7. Threshold Optimization for Best Model
# Find the best non-dummy model by Recall
valid_results = results_df[results_df['Model'] != 'Baseline (Dummy)']
best_row = valid_results.sort_values(by=['Recall', 'PR-AUC'], ascending=[False, False]).iloc[0]

best_ds = best_row['Dataset']
best_model_name = best_row['Model']
best_pipeline = trained_pipelines[best_ds][best_model_name]

print(f"Absolute Best Model: {best_model_name} on {best_ds}")

# Optimize Threshold
y_prob_best = best_pipeline.predict_proba(X_test_raw)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_best)

# We want to maximize Recall, but let's constrain Precision to be at least 0.15 (or use F2 score)
# Using F2 Score to heavily weight Recall over Precision
f2_scores = (5 * precisions * recalls) / (4 * precisions + recalls + 1e-10)
optimal_idx = np.argmax(f2_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Standard Threshold (0.5) Recall: {recall_score(y_test, y_prob_best >= 0.5)}")
print(f"Optimal Threshold ({optimal_threshold:.4f}) Recall: {recall_score(y_test, y_prob_best >= optimal_threshold)}")
print(f"Optimal Threshold Precision: {precision_score(y_test, y_prob_best >= optimal_threshold)}")

Absolute Best Model: Gradient Boosting on Undersampled
Standard Threshold (0.5) Recall: 0.8594164456233422
Optimal Threshold (0.3736) Recall: 0.9336870026525199
Optimal Threshold Precision: 0.41854934601664684


In [10]:
# 8. Cost-Sensitive Analysis (False Positives vs False Negatives)
# Assume the medical cost of missing a cancer diagnosis (False Negative) is $100,000 (legal, life impact)
# Assume the cost of a False Positive is $5,000 (unnecessary biopsy and stress)
COST_FP = 5000
COST_FN = 100000

def calculate_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    fp = cm[0, 1]
    fn = cm[1, 0]
    return (fp * COST_FP) + (fn * COST_FN)

cost_standard = calculate_cost(y_test, y_prob_best >= 0.5)
cost_optimal = calculate_cost(y_test, y_prob_best >= optimal_threshold)
cost_dummy = calculate_cost(y_test, trained_pipelines['Imbalanced']['Baseline (Dummy)'].predict(X_test_raw))

print(f"Total Cost with Baseline (predict all negative): ${cost_dummy:,.2f}")
print(f"Total Cost with Standard 0.5 Threshold: ${cost_standard:,.2f}")
print(f"Total Cost with Optimal Threshold: ${cost_optimal:,.2f}")
print(f"Savings by using Optimal Threshold vs 0.5: ${(cost_standard - cost_optimal):,.2f}")

Total Cost with Baseline (predict all negative): $37,700,000.00
Total Cost with Standard 0.5 Threshold: $7,250,000.00
Total Cost with Optimal Threshold: $4,945,000.00
Savings by using Optimal Threshold vs 0.5: $2,305,000.00


In [11]:
# 9. Save Entire Pipeline and Metadata
metadata = {
    "dataset_version": "v1",
    "training_date": datetime.datetime.now().isoformat(),
    "model_name": best_model_name,
    "dataset_balancing": best_ds,
    "optimal_threshold": float(optimal_threshold),
    "metrics": {
        "recall_optimized": float(recall_score(y_test, y_prob_best >= optimal_threshold)),
        "precision_optimized": float(precision_score(y_test, y_prob_best >= optimal_threshold)),
        "pr_auc": float(best_row['PR-AUC']),
        "roc_auc": float(best_row['ROC-AUC'])
    }
}

joblib.dump(best_pipeline, '../models/pipeline.pkl')
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

# 9b. Experiment Tracking (CSV Log)
log_file = '../reports/experiment_log.csv'
log_record = {
    'Timestamp': metadata['training_date'],
    'Model_Version': metadata['dataset_version'],
    'Model_Name': metadata['model_name'],
    'Dataset_Balancing': metadata['dataset_balancing'],
    'Optimal_Threshold': metadata['optimal_threshold'],
    'Recall_Optimized': metadata['metrics']['recall_optimized'],
    'Precision_Optimized': metadata['metrics']['precision_optimized'],
    'PR_AUC': metadata['metrics']['pr_auc'],
    'ROC_AUC': metadata['metrics']['roc_auc']
}
log_df = pd.DataFrame([log_record])
import os
if not os.path.exists(log_file):
    log_df.to_csv(log_file, index=False)
else:
    log_df.to_csv(log_file, mode='a', header=False, index=False)

print("Saved pipeline.pkl and model_metadata.json to models/")

Saved pipeline.pkl and model_metadata.json to models/


In [12]:
# 10. Local SHAP Explanations Setup
# The backend will need to compute SHAP values for individual patients.
# Since we are using CalibratedClassifierCV, getting direct TreeExplainer is complex.
# We'll save the background dataset for KernelExplainer or just use the base estimator if possible.
# For simplicity in production, we will extract the base estimator from the CalibratedClassifierCV 
# if it's a tree, or just save a small background dataset for KernelExplainer.

background_data = X_train_raw.sample(100, random_state=42)
background_data.to_csv('../models/background_data.csv', index=False)
print("Saved background dataset for SHAP to models/background_data.csv")

Saved background dataset for SHAP to models/background_data.csv
